This notebook will validate metric implementations by comparing results with scipy and sklearn functions.



In [3]:
!pip install pot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 57.1 MB/s eta 0:00:00


In [4]:
# ===========================
# 📌 Import Required Libraries
# ===========================
import numpy as np
import pandas as pd
from metrics import * # Ensure all functions are available
from scipy.spatial.distance import cosine, euclidean, mahalanobis
from scipy.stats import entropy, wasserstein_distance, ks_2samp
from scipy.linalg import pinv
from sklearn.metrics.pairwise import rbf_kernel
import ot


Compute all metric values using customized functions loaded from metrics.py

In [5]:
# ===========================
# 📌 Load Sample Data
# ===========================

IN_FEATURES_PATH = "in_dist_features.csv"
OUT_FEATURES_PATH = "out_dist_features.csv"

# Load in-distribution features (each row represents a feature vector)
train_features = pd.read_csv(IN_FEATURES_PATH, header=None).values  # Shape: (e.g., N_samples, 4096)

# Load out-of-distribution (OOD) features if available
try:
    ood_features = pd.read_csv(OUT_FEATURES_PATH, header=None).values  # Shape: (e.g., M_samples, 4096)
except FileNotFoundError:
    print("⚠️ Warning: OOD feature file not found. Skipping OOD-related computations.")
    ood_features = None

# ===========================
# 📌 Compute Covariance Inverse for Mahalanobis Distance
# ===========================

# Compute the covariance matrix of the in-distribution features
# Adding a small identity matrix ensures numerical stability (regularization)
cov_matrix = np.cov(train_features, rowvar=False) + np.eye(train_features.shape[1]) * 1e-6
# Compute the inverse of the covariance matrix (precompute once)
cov_inv = pinv(cov_matrix)

# ===========================
# 📌 Compute Probability Distributions for KL and JS Divergence
# ===========================
train_probs = np.apply_along_axis(lambda x: x / np.sum(x), 1, train_features)

# ===========================
# 📌 Compute All Metric Values
# ===========================

# 1️⃣ Cosine Similarity: Measures the angular similarity between vectors.
cosine_sim = compute_cosine_similarity(train_features, train_features)

# 2️⃣ Euclidean Distance: Measures the straight-line distance between vectors.
euclidean_sim = compute_euclidean_similarity(train_features, train_features)

# 3️⃣ Mahalanobis Distance: Considers correlations between features for more accurate similarity computation.
mahalanobis_sim = compute_mahalanobis_similarity(train_features, train_features)

# 4️⃣ Kullback-Leibler Divergence: Measures how one probability distribution diverges from another.
kullback_leibler_div = compute_kullback_leibler(train_probs, train_probs)

# 5️⃣ Jensen-Shannon Divergence: Measures probability distribution similarity (symmetric KL divergence).
jensen_shannon_div = compute_jensen_shannon(train_probs, train_probs)

# 6️⃣ Earth Mover’s Distance: Measures the minimum cost required to transform one distribution into another.
earth_movers_dist = compute_earth_movers(train_features, train_features)

# 7️⃣ Maximum Mean Discrepancy (MMD): Measures the discrepancy between distributions in a high-dimensional space.
maximum_mean_discrepancy = compute_maximum_mean_discrepancy(train_features, train_features)

# 8️⃣ Bhattacharyya Distance: Computes distribution overlap, commonly used in classification problems.
bhattacharyya_dist = compute_bhattacharyya(train_probs, train_probs)

# 9️⃣ Entropy: Measures uncertainty/randomness within a distribution.
entropy_vals = compute_entropy(train_probs)

# 🔟 Kolmogorov-Smirnov (KS) Test: Measures the difference between two probability distributions.
kolmogorov_smirnov = compute_kolmogorov_smirnov(train_probs, train_probs)

# 1️⃣1️⃣ Optimal Transport Distance: Measures transformation cost between two distributions.
# Compute centroid (mean distribution)
centroid = convert_to_probability_distribution(np.mean(train_features, axis=0))
# Compute cost matrix based on the centroid distribution
cost_matrix = np.abs(np.subtract.outer(centroid, centroid))  # Square cost matrix
# Iterate through each row of `train_probs` and compute the optimal transport distance
optimal_transport_distances = [
    compute_optimal_transport(train_probs[i], centroid, cost_matrix)
    for i in range(len(train_probs))
]


# 1️⃣2️⃣ Share of Drifted Embedding Components: Detects which feature dimensions exhibit significant changes.
share_drifted = compute_share_of_drifted_components(train_features, train_features)

if ood_features is not None:
    share_drifted_ood = compute_share_of_drifted_components(train_features, ood_features)


Optimal Transport Distances: [1.6529272306733083e-07, 1.2269712472984568e-07, 1.0803365563324161e-07, 2.0335057881359776e-07, 9.137421739777054e-08, 1.8394258230339965e-07, 9.218381300970007e-08, 9.963462248421029e-08, 9.561506870132217e-08, 1.061533200371515e-07, 9.476723755282463e-08, 2.6432099853005e-07, 1.6023932709128737e-07, 1.4525317686374818e-07, 1.015890332328863e-07, 2.125206461768521e-07, 1.7176150394505536e-07, 1.6219419966247155e-07, 1.7757759643378404e-07, 1.6589647502847957e-07, 9.811287080332783e-08, 2.3092393783359114e-07, 2.8493092808042586e-07, 2.388312632810269e-07, 1.5073093098974472e-07, 2.424859708598377e-07, 3.279414376638071e-07, 1.3554204146856113e-07, 3.1840288706238994e-07, 1.6413859758978825e-07, 1.4961821172634812e-07, 1.6087691995635225e-07, 2.347584769080731e-07, 8.728820901421459e-08, 2.210429992782629e-07, 1.7733376478421567e-07, 1.371379631691e-07, 1.1856301140447163e-07, 1.9412993267880828e-07, 1.3403385044589295e-07, 1.2354319941970609e-07, 9.550243

Validate all the computed metrics by comparing them to reference implementations.

In [10]:
# ===========================
# 📌 Compute Centroid (Mean Feature Vector)
# ===========================
# The centroid is the mean of all feature vectors in the dataset.
# It represents the "center" of the feature space and is used as a reference
# for computing similarity metrics like Cosine Similarity, Euclidean Distance, etc.
centroid = np.mean(train_features, axis=0)

# ===========================
# 📌 Precompute Covariance Matrix for Mahalanobis Distance
# ===========================
# The Mahalanobis distance requires the inverse of the covariance matrix.
# Regularization (adding a small identity matrix) ensures numerical stability.
cov_matrix = np.cov(train_features, rowvar=False) + np.eye(train_features.shape[1]) * 1e-6
cov_inv = np.linalg.pinv(cov_matrix)  # Compute the inverse once and reuse it


# ===========================
# 📌 Convert Features to Probability Distributions
# ===========================
# Probability-based metrics like KL Divergence and Jensen-Shannon Divergence
# require the input feature vectors to be normalized as probability distributions.
train_probs = np.apply_along_axis(lambda x: x / np.sum(x), 1, train_features)  # Normalize each row
train_probs = np.array([convert_to_probability_distribution(f) for f in train_features])
centroid_prob = convert_to_probability_distribution(np.mean(train_features, axis=0))

# ===========================
# 📌 Validate Each Metric
# ===========================
print("🔍 Validating Metrics...")

# ✅ Validate Cosine Similarity
# Measures the angular similarity between vectors. Ranges from -1 (opposite) to 1 (identical) usinf scipy cosine
assert np.allclose(
    compute_cosine_similarity(train_features, train_features),
    [1 - cosine(train_features[i], centroid) for i in range(len(train_features))]
)
print("✅ Cosine Similarity validated.")


# ===========================
# ✅ Validate Euclidean Distance
# Measures the straight-line distance between vectors in high-dimensional space using scipy euclidean
assert np.allclose(
    compute_euclidean_similarity(train_features, train_features),
    [euclidean(train_features[i], centroid) for i in range(len(train_features))]
)
print("✅ Euclidean Distance validated.")
# ===========================


# ===========================
# ✅ Validate Mahalanobis Distance
# Considers correlations between features, making it more robust for high-dimensional data using scipy mahalanobis
assert np.allclose(
    compute_mahalanobis_similarity(train_features, train_features),
    [mahalanobis(train_features[i], centroid, cov_inv) for i in range(len(train_features))]
)
print("✅ Mahalanobis Distance validated.")
# ===========================


# ===========================
# ✅ Validate Jensen-Shannon Divergence
# Measures the similarity between two probability distributions using KL divergence in scipy
assert np.allclose(
    compute_jensen_shannon(train_features, train_features),
    [distance.jensenshannon(np.mean(train_features, axis=0) / np.sum(np.mean(train_features, axis=0)),
                            train_features[i] / np.sum(train_features[i]))
     for i in range(len(train_features))]
)
print("✅ Jensen-Shannon Divergence validated.")
# ===========================


# ===========================
# ✅ Validate Kullback-Leibler Divergence
# Normalize input features to ensure they are probability distributions
train_features_normalized = train_features / train_features.sum(axis=1, keepdims=True)
mean_feature_normalized = np.mean(train_features_normalized, axis=0)
# Validate KL divergence computation
assert np.allclose(
    kullback_leibler_div,
    [entropy(train_features_normalized[i], mean_feature_normalized) for i in range(len(train_features_normalized))],
    atol=1e-6,  # Adjust tolerance if necessary
)
print("✅ Kullback-Leibler Divergence validated.")
# ===========================



# ===========================
# ✅ Validate Earth Mover's Distance (Wasserstein Distance)
# Measures the minimum "work" required to transform one distribution into another using scipy
assert np.allclose(
    compute_earth_movers(train_features, train_features),
    [wasserstein_distance(train_features[i], centroid) for i in range(len(train_features))]
)
print("✅ Earth Mover’s Distance validated.")
# ===========================


# ===========================
# ✅ Validate Maximum Mean Discrepancy (MMD)
# Measures how different two distributions are in a high-dimensional space using scipy
# Compute pairwise RBF kernel values for expected MMD
K_XX = rbf_kernel(train_features, train_features)  # Train-to-train
K_YY = rbf_kernel(train_features, train_features)  # Train-to-train (self-pairing)
K_XY = rbf_kernel(train_features, train_features)  # Train-to-train (cross-term)
# Compute expected MMD using the known formula
expected_mmd = np.mean(K_XX) + np.mean(K_YY) - 2 * np.mean(K_XY)

# Validate computed MMD
assert np.allclose(
    compute_maximum_mean_discrepancy(train_features, train_features),
    expected_mmd,
    atol=1e-6  # Adjusted tolerance
), "❌ Maximum Mean Discrepancy validation failed!"

print("✅ Maximum Mean Discrepancy validated.")
# ===========================




# ===========================
# ✅ Validate Entropy
# Measures the randomness or disorder in a distribution.
assert np.allclose(
    compute_entropy(train_features),
    [entropy(train_probs[i]) for i in range(len(train_probs))]
)
print("✅ Entropy validated.")
# ===========================


# ===========================
# ✅ Validate Kolmogorov-Smirnov (KS) Test
# Measures the maximum difference between two cumulative distribution functions.
expected_ks = [
    ks_2samp(convert_to_probability_distribution(train_probs[i]),
             convert_to_probability_distribution(train_probs.mean(axis=0))).statistic
    for i in range(len(train_probs))
]
assert np.allclose(
    compute_kolmogorov_smirnov(train_features, train_features),
    expected_ks,
    atol=1e-3  # Increased tolerance to account for numerical precision errors
)
print("✅ Kolmogorov-Smirnov Test validated successfully.")
# ===========================


# ===========================
# ===========================
# 📌 Validate Optimal Transport Distance
# ===========================
# Convert features into probability distributions
train_probs = np.array([convert_to_probability_distribution(f) for f in train_features])
centroid_prob = convert_to_probability_distribution(np.mean(train_features, axis=0))

# Compute cost matrix based on probability distributions
cost_matrix = np.abs(np.subtract.outer(centroid_prob, centroid_prob))

# Compute expected OT values row-by-row
expected_ot_values = [
    ot.emd2(train_probs[i], centroid_prob, cost_matrix)
    for i in range(len(train_probs))
]

# Validate OT distance row-by-row
assert np.allclose(
    optimal_transport_distances,
    expected_ot_values,
    atol=1e-6  # Set a small absolute tolerance
)

print("✅ Optimal Transport Distance validated successfully.")
# ===========================



# ===========================
# ===========================
# 📌 Validate Share of Drifted Embedding Components
# ===========================
# This metric checks how many feature dimensions exhibit statistically significant changes.
share_drifted = compute_share_of_drifted_components(train_features, train_features)  # Should be ~0
share_drifted_ood = compute_share_of_drifted_components(train_features, ood_features)

# Print results
print(f"Share of Drifted Components (In-Dist vs In-Dist): {share_drifted:.4f}")
print(f"Share of Drifted Components (In-Dist vs OOD): {share_drifted_ood:.4f}")

# Ensure minimal drift within the same distribution
assert share_drifted < 0.05, "❌ Validation Failed: In-distribution should have minimal drift."

print("✅ Share of Drifted Components metric validated successfully!")
# ===========================


# ===========================
# 📌 Final Validation Status
# ===========================
print("\n✅ **All metric implementations validated successfully!**")


🔍 Validating Metrics...
✅ Cosine Similarity validated.
✅ Euclidean Distance validated.
✅ Mahalanobis Distance validated.
✅ Jensen-Shannon Divergence validated.
✅ Kullback-Leibler Divergence validated.
✅ Earth Mover’s Distance validated.
✅ Maximum Mean Discrepancy validated.
✅ Entropy validated.
✅ Kolmogorov-Smirnov Test validated successfully.
✅ Optimal Transport Distance validated successfully.
Share of Drifted Components (In-Dist vs In-Dist): 0.0000
Share of Drifted Components (In-Dist vs OOD): 0.0369
✅ Share of Drifted Components metric validated successfully!

✅ **All metric implementations validated successfully!**
